In [1]:
import numpy as np
import pandas as pd

In [3]:
data = pd.read_csv('https://github.com/dvasiliu/AML/blob/main/Data%20Sets/example_data_classification.csv?raw=true', header=None)

In [4]:
data

,0,1,2
0,34.623660,78.024693,0
1,30.286711,43.894998,0
2,35.847409,72.902198,0
3,60.182599,86.308552,1
4,79.032736,75.344376,1
...,...,...,...
95,83.489163,48.380286,1
96,42.261701,87.103851,1
97,99.315009,68.775409,1
98,55.340018,64.931938,1


In [5]:
x = data.iloc[:,[0,1]].values
y = data.iloc[:,-1].values

## The Goal is to Code LR from Scratch (just do gradient descent)

In [24]:
# let's make a blueprint (a class) that can instantiate objects
class LR:
    def __init__(self):
        self.w = None
        self.b = None

    def sig(self,z):
        return np.clip(1/(1+np.exp(-z)),1e-10,0)
        
    def gradient(self,x,y,w,b):
        errors = y - self.sig(x@w+b)
        return -(errors)@x, -sum(errors)
        
    def fit(self,x,y,lr=0.01,maxiter=1000,bias=True,tol=1e-5):
        # the main goal of this is to update the weights with gradient descent
        # first initialize the weights and the bias term
        self.w = np.random.normal(size=x.shape[1])
        if bias:
            self.b = np.random.normal()
        else:
            self.b = 0
        
        # this is the plain vanilla Gadient Descent (in some references, it is Newton_Raphson)
        
        for _ in range(maxiter):
            gw, gb = self.gradient(x,y,self.w,self.b)
            self.wnew = self.w - lr*gw
            self.bnew = self.b - lr*gb
            if np.linalg.norm(self.wnew-self.w)<tol:
                break
            self.w = self.wnew
            self.b = self.bnew
            
    def predict_proba(self,x):
        self.proba = np.clip(self.sig(x@self.w+self.b),1e-10,0)
        return self.proba

    def predict_classes(self,x,thresh=0.5):
        return (self.predict_proba(x) > thresh) + 0

    def score(self,x,y):
        return 1-sum(abs(y-self.predict_classes(x)))/len(y)
        
        

In [9]:
def zscore(x):
    return (x-np.mean(x,axis=0))/np.std(x,axis=0)

In [25]:
model = LR()

In [26]:
model.fit(x,y,maxiter=1000,lr=0.01)

In [27]:
ypredicted = model.predict_classes(x)

In [29]:
model.score(x,y)

np.float64(0.4)

## Let's compare with LogisticRegression from Scikit_learn

In [20]:
from sklearn.linear_model import LogisticRegression

In [21]:
model2 = LogisticRegression(C=10000)

In [23]:
model2.fit(x,y)
model2.score(x,y)

0.89

## Try RMSPROP to compete with Scikit-Learn

In [94]:
# let's make a blueprint (a class) that can instantiate objects
class LR_rmsprop:
    def __init__(self):
        self.w = None
        self.b = None

    def loss(self,x,y):
        return -np.mean(y*np.log(self.sig(x@self.w+self.b))+(1-y)*np.log(1-self.sig(x@self.w+self.b)))

    def sig(self,z):
        return 1/(1+np.exp(-np.clip(z,-500,500)))
        
    def gradient(self,x,y,w,b):
        errors = y - self.sig(x@w+b)
        return -1/len(x)*(errors)@x, -1/len(x)*sum(errors)
        
    def fit(self,x,y,lr=0.01,maxiter=1000,bias=True,tol=1e-5,beta1=0.9,eps=1e-5,batch_size=32):
        # the main goal of this is to update the weights with gradient descent
        # first initialize the weights and the bias term
        # self.beta1 = beta1
        # self.eps = eps
        self.w = np.random.normal(size=x.shape[1])
        if bias:
            self.b = np.random.normal()
        else:
            self.b = 0
        
        # RMSPROP
        u = 0
        sw = 0
        sb = 0 
        lss = []
        done = False
        for i in range(maxiter):
            minibatches = np.array_split(np.random.permutation(range(len(x))),len(x)//batch_size)
            for batch in minibatches:
                gw, gb = self.gradient(x,y,self.w,self.b)
        
                # here we create adaptive learning rates
                sw = beta1*sw + (1-beta1)*sum(gw**2)
                sb = beta1*sb + (1-beta1)*gb**2
                
                self.wnew = self.w - lr/np.sqrt(sw+eps)*gw
                self.bnew = self.b - lr/np.sqrt(sb+eps)*gb
                u += 1
                lss.append(self.loss(x,y))
                if np.linalg.norm(self.wnew-self.w)<tol:
                    print('The Algorithm has Converged!')
                    done = True
                    break
                self.w = self.wnew
                self.b = self.bnew
                if (u+1)%100 ==0:
                    print(f'After {u+1} updates the Loss is: {self.loss(x,y)}')   
            if done:
                break
            
    def predict_proba(self,x):
        self.proba = self.sig(x@self.w+self.b)
        return self.proba

    def predict_classes(self,x,thresh=0.5):
        return (self.predict_proba(x) > thresh) + 0

    def score(self,x,y):
        return 1-sum(abs(y-self.predict_classes(x)))/len(y)
        
        

In [95]:
model2 = LR_rmsprop()
model2.fit(x,y,lr=0.01,maxiter=1000,batch_size=20)
model2.score(x,y)

After 100 updates the Loss is: 3.0055650120298525
After 200 updates the Loss is: 0.5866572371112617
After 300 updates the Loss is: 0.5672655147125715
After 400 updates the Loss is: 0.5394252645713429
After 500 updates the Loss is: 0.5144128490301076
After 600 updates the Loss is: 0.49191791657610123
After 700 updates the Loss is: 0.471657647144407
After 800 updates the Loss is: 0.4533775341715077
After 900 updates the Loss is: 0.4368507214018084
After 1000 updates the Loss is: 0.4218763977826097
After 1100 updates the Loss is: 0.4082776712289724
After 1200 updates the Loss is: 0.39589922096082153
After 1300 updates the Loss is: 0.38460492794256723
After 1400 updates the Loss is: 0.3742756067027884
After 1500 updates the Loss is: 0.36480690727937753
After 1600 updates the Loss is: 0.356107419072721
After 1700 updates the Loss is: 0.34809698463210687
After 1800 updates the Loss is: 0.3407052170484768
After 1900 updates the Loss is: 0.3338702066821923
After 2000 updates the Loss is: 0.327

/tmp/ipykernel_38496/1621322321.py:8: RuntimeWarning: divide by zero encountered in log
  return -np.mean(y*np.log(self.sig(x@self.w+self.b))+(1-y)*np.log(1-self.sig(x@self.w+self.b)))
/tmp/ipykernel_38496/1621322321.py:8: RuntimeWarning: invalid value encountered in multiply
  return -np.mean(y*np.log(self.sig(x@self.w+self.b))+(1-y)*np.log(1-self.sig(x@self.w+self.b)))


np.float64(0.88)

In [89]:
# RMSPROP approach
class LR_rmsprop:
    def __init__(self):
        self.w = None
        self.b = None

    def loss(self,x,y):
        predictions = self.sig(x@self.w+self.b)
        log_lik = np.sum(y * np.log(predictions) + (1 - y) * np.log(1 - predictions))
        return -log_lik

    def sig(self,z):
        return 1/(1+np.exp(-z))
        
    def gradient(self,x,y):
        errors = y - self.sig(x@self.w+self.b)
        return -1/len(x)*(errors)@x, -1/len(x)*sum(errors)
        
    def fit(self,x,y,lr=0.01,maxiter=1000,intercept=True,tol=1e-5,sw=0,sb=0,eps=1e-6,batch_size=32,done=False,beta1=0.9,u=0):
        # the main goal of this is to update the weights with gradient descent
        # first initialize the weights and the bias term
        self.w = np.random.normal(size=x.shape[1])
        if intercept:
            self.b = np.random.normal()
        else:
            self.b = 0
        lss= []
        for i in range(maxiter):
            minibatches = np.array_split(np.random.permutation(range(len(x))),len(x)//batch_size)
            for batch in minibatches:
                gw, gb = self.gradient(x,y)
        
                # here we create adaptive learning rates
                sw = beta1*sw + (1-beta1)*sum(gw**2)
                sb = beta1*sb + (1-beta1)*gb**2
                
                self.wnew = self.w - lr/np.sqrt(sw+eps)*gw
                self.bnew = self.b - lr/np.sqrt(sb+eps)*gb
                u += 1
                lss.append(self.loss(x,y))
                if np.linalg.norm(self.wnew-self.w)<tol:
                    print('The Algorithm has Converged!')
                    done = True
                    break
                if (u+1)%100 ==0:
                    print(f'After {u+1} updates the Loss is: {self.loss(x,y)}')
                self.w = self.wnew
                self.b = self.bnew
            if done:
                break
            
    def predict_proba(self,x):
        return self.sig(x@self.w+self.b)

    def predict_classes(self,x,thresh=0.5):
        return (self.sig(x@self.w+self.b)>thresh) + 0 

    def score(self,x,y):
        return 1-sum(abs(y-self.predict_classes(x)))/len(y)

In [85]:
model2.w

array([0.07284948, 0.06614962])

In [86]:
# for comparison (no performance difference, just a different coding approach)
# Method 2: From scratch
class LogisticRegressionScratch:
    def __init__(self, learning_rate=0.01, iterations=1000):
        self.lr = learning_rate
        self.iterations = iterations
        self.weights = None
        self.bias = None
    
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
    
    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.weights = np.zeros(n_features)
        self.bias = 0
        
        # Gradient descent
        for _ in range(self.iterations):
            linear_pred = np.dot(X, self.weights) + self.bias
            predictions = self.sigmoid(linear_pred)
            
            # Compute gradients
            dw = (1/n_samples) * np.dot(X.T, (predictions - y))
            db = (1/n_samples) * np.sum(predictions - y)
            
            # Update parameters
            self.weights -= self.lr * dw
            self.bias -= self.lr * db
    
    def predict(self, X):
        linear_pred = np.dot(X, self.weights) + self.bias
        y_pred = self.sigmoid(linear_pred)
        return [1 if i > 0.5 else 0 for i in y_pred]

In [70]:
model2 = LogisticRegressionScratch()

In [71]:
model2.fit(x,y)

In [73]:
from sklearn.linear_model import LogisticRegression

In [74]:
model3 = LogisticRegression()

In [75]:
model3.fit(x,y)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,None
,solver,'lbfgs'
,max_iter,100
,multi_class,'deprecated'


In [76]:
model3.predict(x)

array([0, 0, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1,
       0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 0, 1, 1,
       0, 0, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 1])

In [25]:
model2 = LogisticRegressionMLE_Improved(regularization=1.0)

In [26]:
model2.fit(zscore(x),y)

Optimization successful: True
Final negative log-likelihood: 27.9796
Number of iterations: 9


In [20]:
sum(abs(y-model2.predict(zscore(x))))/len(y)

np.float64(0.11)